### Baseline: RNN-AVG (Average Pooling)

The simplest baseline for sentence representation is **average pooling** over the
BLSTM hidden states.  Given the hidden state sequence $\{h_1, h_2, \dots, h_L\}$
produced by the shared BLSTM encoder, the sentence representation is simply:

$$r = \frac{1}{L} \sum_{p=1}^{L} h_p$$

This is applied independently to both the question and the answer.

**Why use this as a baseline?**
- It provides a *lower bound* on performance: there is no learned weighting of
  positions, no attention, and no positional information.
- Comparing RNN-AVG → RNN-ATT shows the benefit of **attention**.
- Comparing RNN-ATT → RNN-POA shows the additional benefit of **positional attention**.

**How it differs from the other models:**

| Model   | Question repr. | Answer repr.          | Learned weighting? | Position-aware? |
|---------|----------------|-----------------------|--------------------|-----------------|
| RNN-AVG | avg-pool       | avg-pool              | No                 | No              |
| RNN-ATT | avg-pool       | attention-weighted     | Yes                | No              |
| RNN-POA | avg-pool       | positional-attention   | Yes                | Yes             |

Note: the `SharedBLSTM` encoder already computes a mean-pooled representation (`pooled`)
as one of its outputs, so RNN-AVG simply uses that directly for both question and answer.

In [ ]:
# ── Baseline: Average-Pooling BLSTM (no attention at all) ──
class AvgPoolBLSTM(nn.Module):
    """
    Baseline: RNN-AVG — average pooling over BLSTM hidden states.

    Both question and answer representations are computed as:
        r = (1/L) * sum(h_i)  for i = 1..L  (valid, non-padded positions)

    This is the simplest baseline: no attention mechanism, no positional
    information.  SharedBLSTM already computes this mean-pooled output,
    so we just use it directly.

    Similarity is computed via Manhattan distance:
        sim(q, a) = exp(-||r_q - r_a||_1)
    """
    def __init__(self, embed_matrix, hidden_dim=50, dropout=0.2):
        super().__init__()
        self.encoder = SharedBLSTM(embed_matrix, hidden_dim, dropout)

    def forward(self, q_ids, a_ids, q_len, a_len, q_pos):
        """
        Args:
            q_ids:  (B, L_q) question token ids
            a_ids:  (B, L_a) answer token ids
            q_len:  (B,) question lengths
            a_len:  (B,) answer lengths
            q_pos:  list of lists – UNUSED (kept for interface compatibility)

        Returns:
            sim: (B,) similarity scores in (0, 1]
        """
        # ── Encode question and answer ──
        _, q_pooled = self.encoder(q_ids, q_len)   # (B, 2H)  mean-pooled
        _, a_pooled = self.encoder(a_ids, a_len)   # (B, 2H)  mean-pooled

        # ── Manhattan distance similarity ──
        # sim(q, a) = exp(-||r_q - r_a||_1)
        sim = torch.exp(-torch.sum(torch.abs(q_pooled - a_pooled), dim=1))  # (B,)
        return sim

In [ ]:
class AttentionBLSTM(nn.Module):
    """
    Baseline: classical attention-based BLSTM (no positional influence).
    Used for comparison to show the benefit of positional attention.
    """
    def __init__(self, embed_matrix, hidden_dim=50, dropout=0.2):
        super().__init__()
        self.encoder = SharedBLSTM(embed_matrix, hidden_dim, dropout)
        self.W = nn.Linear(2*hidden_dim, 2*hidden_dim, bias=False)

    def forward(self, q_ids, a_ids, q_len, a_len, q_pos):
        _, q_pooled   = self.encoder(q_ids, q_len)
        a_hidden, _   = self.encoder(a_ids, a_len)

        Wq = self.W(q_pooled)
        scores = torch.bmm(a_hidden, Wq.unsqueeze(2)).squeeze(2)
        a_mask = (a_ids != 0).float()
        scores = scores.masked_fill(a_mask == 0, -1e9)
        alpha  = F.softmax(scores, dim=1)
        a_attended = torch.bmm(alpha.unsqueeze(1), a_hidden).squeeze(1)

        sim = torch.exp(-torch.sum(torch.abs(q_pooled - a_attended), dim=1))
        return sim


### Evaluation Metrics: MAP & MRR

**Mean Average Precision (MAP)**: For each question, compute average precision
over its ranked candidate answers, then average across all questions.

**Mean Reciprocal Rank (MRR)**: For each question, find the rank of the first
correct answer, take its reciprocal, then average across all questions.

In [ ]:
def compute_metrics(qid_list, scores, labels):
    """
    Compute MAP and MRR for answer selection.

    Groups predictions by question ID, ranks answers by predicted score,
    and computes MAP and MRR.

    Args:
        qid_list: list of question IDs (one per sample)
        scores:   np.array of predicted similarity scores
        labels:   np.array of ground-truth binary labels

    Returns:
        (MAP, MRR) tuple
    """
    # Group by question ID
    groups = defaultdict(list)
    for qid, score, label in zip(qid_list, scores, labels):
        groups[qid].append((score, label))

    avg_precisions = []
    reciprocal_ranks = []

    for qid, pairs in groups.items():
        # Sort by score descending
        pairs.sort(key=lambda x: x[0], reverse=True)
        sorted_labels = [p[1] for p in pairs]

        # Skip questions with no positive answer
        if sum(sorted_labels) == 0:
            continue

        # Average Precision
        num_correct = 0
        precision_sum = 0.0
        for rank, lbl in enumerate(sorted_labels, 1):
            if lbl == 1:
                num_correct += 1
                precision_sum += num_correct / rank
        ap = precision_sum / num_correct
        avg_precisions.append(ap)

        # Reciprocal Rank
        for rank, lbl in enumerate(sorted_labels, 1):
            if lbl == 1:
                reciprocal_ranks.append(1.0 / rank)
                break

    MAP = np.mean(avg_precisions) if avg_precisions else 0.0
    MRR = np.mean(reciprocal_ranks) if reciprocal_ranks else 0.0
    return MAP, MRR




In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    """Run model on a data loader and return loss, MAP, MRR."""
    model.eval()
    all_scores, all_labels, all_qids = [], [], []
    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        q_ids = batch['q_ids'].to(device)
        a_ids = batch['a_ids'].to(device)
        q_len = batch['q_len'].to(device)
        a_len = batch['a_len'].to(device)
        labels = batch['label'].to(device)

        sim = model(q_ids, a_ids, q_len, a_len, batch['q_pos'])
        sim = sim.clamp(1e-7, 1 - 1e-7)  # numerical safety
        loss = F.binary_cross_entropy(sim, labels)
        total_loss += loss.item()
        n_batches += 1

        all_scores.extend(sim.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())
        all_qids.extend(batch['qid'])

    avg_loss = total_loss / max(n_batches, 1)
    MAP, MRR = compute_metrics(all_qids, np.array(all_scores), np.array(all_labels))
    return avg_loss, MAP, MRR

### Training Loop

- **Loss**: Binary cross-entropy (relevant=1, irrelevant=0)
- **Optimizer**: Adadelta (as per the paper)
- **Early stopping**: based on dev MAP

In [ ]:
def train_model(model, train_loader, dev_loader, n_epochs=30, patience=5, lr=1.0,
                save_name='best_model.pt'):
    """
    Train the model with Adadelta and early stopping.

    Returns:
        history dict with train/dev losses, MAP, MRR per epoch
    """
    optimizer = torch.optim.Adadelta(model.parameters(), lr=lr)
    best_dev_map = 0.0
    patience_counter = 0
    history = {'train_loss': [], 'dev_loss': [], 'dev_map': [], 'dev_mrr': []}

    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0.0
        n_batches = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{n_epochs}', leave=False)
        for batch in pbar:
            q_ids = batch['q_ids'].to(device)
            a_ids = batch['a_ids'].to(device)
            q_len = batch['q_len'].to(device)
            a_len = batch['a_len'].to(device)
            labels = batch['label'].to(device)

            optimizer.zero_grad()
            sim = model(q_ids, a_ids, q_len, a_len, batch['q_pos'])
            sim = sim.clamp(1e-7, 1 - 1e-7)  # prevent log(0)

            # Binary cross-entropy loss
            loss = F.binary_cross_entropy(sim, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1
            pbar.set_postfix(loss=f'{loss.item():.4f}')

        avg_train_loss = epoch_loss / max(n_batches, 1)

        # ── Dev evaluation ──
        dev_loss, dev_map, dev_mrr = evaluate(model, dev_loader)

        history['train_loss'].append(avg_train_loss)
        history['dev_loss'].append(dev_loss)
        history['dev_map'].append(dev_map)
        history['dev_mrr'].append(dev_mrr)

        print(f'Epoch {epoch:2d} | '
              f'Train Loss: {avg_train_loss:.4f} | '
              f'Dev Loss: {dev_loss:.4f} | '
              f'Dev MAP: {dev_map:.4f} | '
              f'Dev MRR: {dev_mrr:.4f}')

        # Early stopping on dev MAP
        if dev_map > best_dev_map:
            best_dev_map = dev_map
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pt')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch} (no MAP improvement for {patience} epochs)')
                break

    # Load best model
    model.load_state_dict(torch.load('best_model.pt', weights_only=True))
    return history